#Лабораторная работа №1
##Классическое детектирование объектов:каскад Виолы–Джонса и HOG+SVM.
#
В работе сравниваются два классических подхода к детектированию объектов:

1. каскадный классификатор Виолы - Джонса;
2. HOG-дескрипторы и линейный SVM.

Этот notebook содержит только заготовки для организации эксперимента и примеры работы с библиотеками OpenCV и scikit-learn. Подготовку данных, обучение собственного каскада, реализацию детектора HOG+SVM и расчёт итоговых метрик необходимо выполнить самостоятельно.

Подключим библиотеки, зафиксируем начальное значение генераторов случайных чисел и зададим основные пути. Значение DATA_DIR можно изменить в соответствии со структурой проекта.

In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import LinearSVC


SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data/lr01")

POSITIVE_DIR = DATA_DIR / "positive"
NEGATIVE_DIR = DATA_DIR / "negative"
TEST_DIR = DATA_DIR / "test"
ANNOTATIONS_DIR = DATA_DIR / "annotations"

DEMO_IMAGE = TEST_DIR / "demo.jpg"

print("Версия OpenCV:", cv2.__version__)
print("Каталог с данными:", DATA_DIR.resolve())

## Рекомендуемая структура данных

Для основной части работы удобно заранее разделить изображения на положительные, отрицательные и тестовые. Разметку тестовых изображений лучше хранить отдельно.
```text
data/lr01/
├── positive/
├── negative/
├── test/
└── annotations/
 ```
 В шаблоне рассматривается детектирование лиц на FDDB. Все методы должны проверяться на одном и том же наборе тестовых изображений.

FDDB задаёт положение лиц эллипсами. Для расчёта IoU и отображения результатов эллиптическую разметку необходимо преобразовать в охватывающие прямоугольники. Способ преобразования нужно описать в отчёте и одинаково применять ко всем методам.

Следующие функции отвечают только за чтение изображений и визуализацию прямоугольных рамок.
Их можно использовать при проверке обоих детекторов

In [ ]:
def read_image(path: Path) -> np.ndarray:
    """Чтение цветного изображения средствами OpenCV."""
    image = cv2.imread(str(path))

    if image is None:
        raise FileNotFoundError(
            f"Не удалось прочитать изображение: {path}"
        )

    return image


def draw_boxes(
    image_bgr: np.ndarray,
    boxes_xywh: list[tuple[int, int, int, int]],
) -> np.ndarray:
    """Нанесение рамок в формате x, y, width, height."""
    result = image_bgr.copy()

    for x, y, width, height in boxes_xywh:
        cv2.rectangle(
            result,
            (x, y),
            (x + width, y + height),
            color=(0, 255, 0),
            thickness=2,
        )

    return result


def show_bgr(
    image_bgr: np.ndarray,
    title: str = "",
) -> None:
    """Вывод BGR-изображения через matplotlib."""
    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )

    plt.figure(figsize=(8, 5))
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1. Проверка готового каскада OpenCV

Сначала проверим стандартный каскад лиц из состава OpenCV. Этот пример знакомит с CascadeClassifier и detectMultiScale, но не заменяет обучение собственного каскада.

In [ ]:
cascade_path = (
    Path(cv2.data.haarcascades)
    / "haarcascade_frontalface_default.xml"
)

haar_cascade = cv2.CascadeClassifier(
    str(cascade_path)
)

if haar_cascade.empty():
    raise RuntimeError(
        f"Каскад не загружен: {cascade_path}"
    )


def detect_with_pretrained_haar(
    image_bgr: np.ndarray,
    scale_factor: float = 1.1,
    min_neighbors: int = 5,
    min_size: tuple[int, int] = (30, 30),
) -> list[tuple[int, int, int, int]]:
    """Запуск стандартного каскада лиц OpenCV."""
    gray = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2GRAY,
    )

    gray = cv2.equalizeHist(gray)

    detections = haar_cascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=min_size,
    )

    return [
        tuple(map(int, box))
        for box in detections
    ]

На одном изображении можно проверить, что каскад загружается, возвращает рамки и корректно работает с функциями визуализации.

In [ ]:
if DEMO_IMAGE.exists():
    demo_image = read_image(DEMO_IMAGE)

    demo_boxes = detect_with_pretrained_haar(
        demo_image
    )

    print("Количество детекций:", len(demo_boxes))

    result_image = draw_boxes(
        demo_image,
        demo_boxes,
    )

    show_bgr(
        result_image,
        "Проверка стандартного каскада OpenCV",
    )
else:
    print("Добавьте тестовое изображение:")
    print(DEMO_IMAGE.resolve())

Обычный вызов detectMultiScale возвращает только координаты рамок. Для построения Precision–Recall curve каждой детекции потребуется числовая оценка.

Изучите метод detectMultiScale3, параметр outputRejectLevels и возвращаемые значения levelWeights. Функция ниже должна возвращать два объекта:

- boxes - координаты рамок;
- scores - оценки детекций.

In [ ]:
def detect_with_pretrained_haar_scored(
    image_bgr: np.ndarray,
):
    """Получение рамок и оценок готового каскада."""
    raise NotImplementedError(
        "Функция заполняется при выполнении лабораторной работы"
    )

## 2. Подготовка собственного каскада
Для обучения собственного каскада используются утилиты opencv_createsamples и opencv_traincascade.


Для собственного каскада необходимо подготовить описание положительных и отрицательных изображений, создать вектор обучающих примеров и запустить opencv_traincascade.

В следующей ячейке приведена только форма команд. Число примеров, размер окна, количество стадий и другие параметры следует выбрать самостоятельно и обосновать в отчёте.

In [ ]:
createsamples_command = [
    "opencv_createsamples",
    "-info", "path/to/positive.txt",
    "-vec", "path/to/samples.vec",
    "-num", "NUM_SAMPLES",
    "-w", "WINDOW_WIDTH",
    "-h", "WINDOW_HEIGHT",
]

traincascade_command = [
    "opencv_traincascade",
    "-data", "path/to/cascade",
    "-vec", "path/to/samples.vec",
    "-bg", "path/to/negative.txt",
    "-numPos", "NUM_POSITIVE",
    "-numNeg", "NUM_NEGATIVE",
    "-numStages", "NUM_STAGES",
    "-w", "WINDOW_WIDTH",
    "-h", "WINDOW_HEIGHT",
]

print("Шаблон подготовки примеров:")
print(" ".join(createsamples_command))

print("\nШаблон обучения каскада:")
print(" ".join(traincascade_command))

## 4. HOG-дескрипторы

Для согласования с каскадом лиц используется квадратное окно 64 × 64.

OpenCV принимает размеры как (width, height), тогда как форма массива NumPy записывается как (height, width). Это различие следует учитывать при изменении параметров.

In [ ]:
HOG_WINDOW = (64, 64)

hog = cv2.HOGDescriptor(
    HOG_WINDOW,
    (16, 16),
    (8, 8),
    (8, 8),
    9,
)


def extract_hog(
    gray_window: np.ndarray,
) -> np.ndarray:
    """Получение HOG-признаков для одного окна."""
    if gray_window.ndim != 2:
        raise ValueError(
            "Ожидается одноканальное изображение"
        )

    resized = cv2.resize(
        gray_window,
        HOG_WINDOW,
    )

    descriptor = hog.compute(resized)

    return descriptor.reshape(-1)


print(
    "Размер HOG-вектора:",
    hog.getDescriptorSize(),
)

Следующая функция должна прочитать положительные и отрицательные
изображения, выделить окна, получить HOG-признаки и сформировать
массивы `X` и `y`.

Изображения из тестовой части FDDB нельзя использовать при обучении
классификатора.

In [ ]:
def build_window_dataset(
    positive_dir: Path,
    negative_dir: Path,
) -> tuple[np.ndarray, np.ndarray]:
    """Формирование выборки HOG-признаков и меток классов."""
    raise NotImplementedError(
        "Функция заполняется при выполнении лабораторной работы"
    )

## 4. Обучение линейного SVM
#
После формирования `X` и `y` данные необходимо разделить на
обучающую и валидационную части.
#
Окна, полученные из одного исходного изображения, желательно относить
только к одной части выборки. Это снижает риск утечки данных.

In [ ]:
svm = LinearSVC(
    C=1.0,
    random_state=SEED,
    max_iter=10_000,
)

# X, y = build_window_dataset(
#     POSITIVE_DIR,
#     NEGATIVE_DIR,
# )

# Выполните разбиение данных.
# Обучите классификатор на обучающей части.
# Проверьте качество на отложенных окнах.

print("Классификатор создан, но ещё не обучен")

Для итогового детектора недостаточно использовать
`classifier.predict`, поскольку этот метод возвращает только метки
классов.

Для сортировки детекций и построения Precision–Recall curve следует
bспользовать значения `classifier.decision_function(...)`.

## 5. Многомасштабный HOG+SVM-детектор

После обучения оконного классификатора изображение обрабатывается
в нескольких масштабах. На каждом уровне используется скользящее
окно.

Следующие функции относятся к основной части лабораторной работы
и оставлены незаполненными.

In [ ]:
def image_pyramid(
    image: np.ndarray,
    scale: float,
):
    """Построение пирамиды изображений."""
    raise NotImplementedError


def sliding_window(
    image: np.ndarray,
    window_size: tuple[int, int],
    step: int,
):
    """Перебор окон заданного размера."""
    raise NotImplementedError


def detect_with_hog_svm(
    image_bgr: np.ndarray,
    classifier: LinearSVC,
):
    """
    Получение детекций HOG+SVM.

    Функция должна возвращать:
    - boxes - координаты рамок;
    - scores - значения decision_function.
    """
    raise NotImplementedError

## 6. Удаление дублирующихся детекций

Несколько соседних окон могут соответствовать одному объекту. Для удаления дублирующихся рамок требуется реализовать non-maximum suppression.

In [ ]:
def intersection_over_union(
    box_a,
    box_b,
) -> float:
    """Вычисление IoU для двух рамок."""
    raise NotImplementedError(
        "Допишите вычисление площади пересечения и объединения"
    )


def non_max_suppression(
    boxes,
    scores,
    iou_threshold: float,
):
    """Удаление дублирующихся детекций."""
    raise NotImplementedError(
        "Допишите алгоритм NMS"
    )

## 7. Правила оценки качества

Для оконного классификатора рассчитываются:

- TPR;
- FPR;
- Precision;
- Recall.

Здесь отрицательными примерами являются фиксированные фоновые окна,
поэтому число `TN` определено.

Для итогового детектора рассчитываются:

- Precision;
- Recall;
- AP;
- среднее количество ложных срабатываний на изображение при
  необходимости.

В работе рассматривается один класс - лицо. Поэтому `mAP` совпадает
с AP этого класса. В отчёте следует указывать порог IoU.

Для оценки детектора предсказанные рамки сопоставляются с эталонными
рамками по порогу IoU.

Каждая истинная рамка может быть сопоставлена не более чем с одной
детекцией.

In [ ]:
def match_detections_to_ground_truth(
    predicted_boxes,
    ground_truth_boxes,
    iou_threshold: float = 0.5,
):
    """Сопоставление предсказанных и истинных рамок."""
    raise NotImplementedError(
        "Допишите правила определения TP, FP и FN"
    )


def evaluate_detector(
    predictions,
    annotations,
):
    """Расчёт показателей качества детектора."""
    raise NotImplementedError(
        "Допишите TPR, FPR, Precision, Recall и AP/mAP"
    )

## 8. Измерение скорости

Итоговый FPS следует измерять на наборе тестовых изображений после
нескольких прогревочных запусков.

Для честного сравнения следует измерять полный процесс: подготовку
изображения, детектирование и NMS

In [ ]:
def measure_fps(
    detector,
    image_bgr: np.ndarray,
    repeats: int = 20,
) -> float:
    """Измерение средней скорости работы детектора."""
    if repeats <= 0:
        raise ValueError(
            "Число повторов должно быть положительным"
        )

    start_time = perf_counter()

    for _ in range(repeats):
        detector(image_bgr)

    elapsed = perf_counter() - start_time

    return repeats / elapsed

## Что должно быть выполнено в основной части

### Каскад Виолы - Джонса

- подготовка положительных и отрицательных изображений;
- создание обучающего вектора;
- обучение собственного каскада;
- подбор параметров;
- проверка на независимой выборке;
- расчёт TPR, FPR, Precision, Recall и FPS;
- разбор верных и ошибочных детекций.

### HOG+SVM

- формирование положительных и отрицательных окон;
- извлечение HOG-признаков;
- обучение LinearSVC;
- реализация пирамиды изображений;
- реализация скользящего окна;
- применение NMS;
- построение Precision–Recall curve;
- расчёт AP или mAP;
- анализ ошибок.

### Сравнение

Оба метода должны проверяться на одной тестовой выборке. В выводах следует сопоставить качество, скорость, устойчивость к масштабу и типичные причины ошибок.

## Пример итоговой таблицы

| Метод | TPR | FPR | Precision | Recall | mAP | FPS |
|---|---:|---:|---:|---:|---:|---:|
| Стандартный каскад OpenCV |  |  |  |  |  |  |
| Собственный каскад |  |  |  |  |  |  |
| HOG+SVM |  |  |  |  |  |  |

#Правила использования внешних ресурсов
- Допускается использование OpenCV, scikit-learn, NumPy
- Запрещено использование готовых реализаций из сторонних источников (кроме стандартных библиотек)
- Использование LLM (ChatGPT, Copilot) - только для пояснения кода, не для генерации готового решения